In [ ]:
#!pip install openai pydub


In [3]:
import os
import io
import json
import math
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from pydub import AudioSegment

load_dotenv()

# --- API client ---
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# --- Directory structure ---
for folder in ["audio", "audio/chunks", "transcripts"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("✅ Environment ready.")
print(f"📁 Directories created: audio/, audio/chunks/, transcripts/")

✅ Environment ready.
📁 Directories created: audio/, audio/chunks/, transcripts/


### Step 2 Sample Meeting Audio

In [7]:
# Update this path to point to your audio file
AUDIO_FILE = "audio/meeting_sample.mp3"  # Supports mp3, wav, m4a, ogg, webm

# Load with pydub
audio = AudioSegment.from_file(AUDIO_FILE)

duration_minutes = len(audio) / 1000 / 60
print(f"✅ Loaded: {AUDIO_FILE}")
print(f"   Duration : {duration_minutes:.1f} minutes ({len(audio)/1000:.1f} seconds)")
print(f"   Channels : {audio.channels}")
print(f"   Frame rate: {audio.frame_rate} Hz")
print(f"   Sample width: {audio.sample_width} bytes")

✅ Loaded: audio/meeting_sample.mp3
   Duration : 24.1 minutes (1447.3 seconds)
   Channels : 1
   Frame rate: 44100 Hz
   Sample width: 2 bytes


### Step 3: Basic Transcription (Without Chunking)

In [8]:
def transcribe_file(file_path: str, prompt: str = None) -> object:
    """Transcribe an audio file with the Whisper API.
    
    Args:
        file_path: Path to the audio file (max 25 MB).
        prompt:    Optional context string to guide transcription.

    Returns:
        Whisper transcript object with .text and optional .segments / .words.
    """
    kwargs = dict(
        model="whisper-1",
        response_format="verbose_json",
        timestamp_granularities=["segment", "word"]
    )
    if prompt:
        kwargs["prompt"] = prompt

    with open(file_path, "rb") as f:
        transcript = client.audio.transcriptions.create(file=f, **kwargs)
    return transcript


# Basic transcription — no prompt
print("🤖 Transcribing (no prompt)...")
basic_transcript = transcribe_file(AUDIO_FILE)

print("\n📝 Transcription result:")
print("-" * 50)
print(basic_transcript.text)
print("-" * 50)
print(f"\n🌍 Language detected : {basic_transcript.language}")
print(f"⏱️  Duration          : {basic_transcript.duration:.2f}s")

🤖 Transcribing (no prompt)...

📝 Transcription result:
--------------------------------------------------
Ladies and gentlemen, thank you for standing by and welcome to the Atrium 2020 year-end report. At this time, all participants are in a listen-only mode. After the speaker's presentation, there will be a question and answer session. To ask a question during the session, you will need to press star 1 on your telephone keypad. If you require any further assistance, please press star 0. Please be advised that today's conference is being recorded. I would now like to hand the conference over to Marco Antonio Northland, CEO. Please go ahead. Thank you, Michelle. I would like to welcome you to Atrium's 2020 fourth quarter and four-year operational finance update call. I'm here today with Cristian La Cueva, our Chief Financial Officer, Martin Orovic, our Chief Investment Officer, and David Ramirez, our Director of Accounting and Reporting. I will start today's call by taking you through t

### Step 4: Transcription with Prompts (Guided Approach)

In [10]:
# ---- Customize this prompt for your meeting content ----
MEETING_PROMPT = (
    "This is a business meeting transcript. "
    "Participants: Marco Antonio Northland, CEO; Michelle, Operator, Cristian La Puebla, CFO. "
    "Business terms: Etrion, EBITDA, runway, burn rate, pivot, go-to-market strategy. "
    "COVID, funding, and product launch are likely topics. "
)

print("🤖 Transcribing with context prompt...")
guided_transcript = transcribe_file(AUDIO_FILE, prompt=MEETING_PROMPT)

print("\n📝 Guided transcription:")
print("-" * 50)
print(guided_transcript.text)
print("-" * 50)

🤖 Transcribing with context prompt...

📝 Guided transcription:
--------------------------------------------------
Ladies and gentlemen, thank you for standing by and welcome to the Etrion 2020 year-end report. At this time, all participants are in a listen-only mode. After the speaker's presentation, there will be a question and answer session. To ask a question during the session, you will need to press star 1 on your telephone keypad. If you require any further assistance, please press star 0. Please be advised that today's conference is being recorded. I would now like to hand the conference over to Marco Antonio Northland, CEO. Please go ahead. Thank you, Michelle. I would like to welcome you to Etrion's 2020 fourth quarter and four-year operational finance update call. I'm here today with Cristian La Puebla, our Chief Financial Officer, Martin Orebic, our Chief Investment Officer, and David Ramirez, our Director of Accounting and Reporting. I will start today the call by taking yo

### Step 5: Transcription Without Prompts (Unguided Approach)

In [11]:
def word_set(text: str) -> set:
    """Return lowercase word tokens from text."""
    return set(text.lower().split())


unguided_words = word_set(basic_transcript.text)
guided_words   = word_set(guided_transcript.text)

only_in_guided   = guided_words - unguided_words
only_in_unguided = unguided_words - guided_words

print("🔍 Comparison: Guided vs. Unguided\n")
print(f"Unguided word count : {len(unguided_words)}")
print(f"Guided   word count : {len(guided_words)}")
print(f"\nWords only in GUIDED   : {only_in_guided if only_in_guided else '(none)'}")
print(f"Words only in UNGUIDED : {only_in_unguided if only_in_unguided else '(none)'}")

print("\n--- Unguided ---")
print(basic_transcript.text)
print("\n--- Guided ---")
print(guided_transcript.text)

🔍 Comparison: Guided vs. Unguided

Unguided word count : 984
Guided   word count : 976

Words only in GUIDED   : {'hire', 'distributions.', "shareholders'", 'tohoku', 'etrion', 'orebic,', 'sweden', 'claimants.', 'detail', 'cost-related', 'pinel', 'near', 'megawatts', 'system', '15', 'puebla,', 'full', 'balanchi', 'hedge', 'etron', 'thing,', '14,', 'in,', 'his', 'continued', 'q4,', 'segments', 'electron', 'note', 'liability,', 'operations,', 'looking', '11', 'installed', '5,', "funds'", 'network', 'shown', 'uk', 'symbol', '8.9', 'megawatt', 'frs', 'please?', "etrion's", 'operator,', 'write-off', 'covid.', 'time', 'manager', 'elements.', 'expects'}
Words only in UNGUIDED : {'google,', 'us', 'yokaichi', 'operator.', 'mean,', 'second-level', 'hired', 'write-offs', 'resolution.', 'orovic,', 'please.', 'sweden,', 'notes', 'how', '60-megawatt', 'drop', 'hope', 'planning', 'site', 'sorry.', 'provides', 'italy?', 'atrium', 'notice', 'do.', 'p&l', 'elements,', 'q4.', 'networks', 'veterans', 'ope

### Step 6: Implementing Audio Chunking

In [12]:
def chunk_audio(audio_path: str,
                chunk_minutes: int = 10,
                output_dir: str = "audio/chunks") -> list[dict]:
    """Split audio into fixed-duration chunks.

    Args:
        audio_path   : Path to the source audio file.
        chunk_minutes: Length of each chunk in minutes.
        output_dir   : Directory to write chunk files.

    Returns:
        List of dicts with keys 'path', 'start_ms', 'end_ms', 'index'.
    """
    audio     = AudioSegment.from_file(audio_path)
    chunk_ms  = chunk_minutes * 60 * 1000
    n_chunks  = math.ceil(len(audio) / chunk_ms)
    chunks    = []
    stem      = Path(audio_path).stem

    for i in range(n_chunks):
        start_ms = i * chunk_ms
        end_ms   = min(start_ms + chunk_ms, len(audio))
        segment  = audio[start_ms:end_ms]

        out_path = f"{output_dir}/{stem}_chunk{i+1:02d}.mp3"
        segment.export(out_path, format="mp3")

        chunks.append({
            "index"   : i,
            "path"    : out_path,
            "start_ms": start_ms,
            "end_ms"  : end_ms,
        })
        print(f"  Chunk {i+1:02d}: {start_ms/1000:.0f}s – {end_ms/1000:.0f}s → {out_path}")

    print(f"\n✅ Created {n_chunks} chunk(s) of ≤ {chunk_minutes} minutes each.")
    return chunks


CHUNK_MINUTES = 10   # Adjust for your recordings
print(f"🔪 Splitting audio into {CHUNK_MINUTES}-minute chunks...\n")
chunks = chunk_audio(AUDIO_FILE, chunk_minutes=CHUNK_MINUTES)

🔪 Splitting audio into 10-minute chunks...

  Chunk 01: 0s – 600s → audio/chunks/meeting_sample_chunk01.mp3
  Chunk 02: 600s – 1200s → audio/chunks/meeting_sample_chunk02.mp3
  Chunk 03: 1200s – 1447s → audio/chunks/meeting_sample_chunk03.mp3

✅ Created 3 chunk(s) of ≤ 10 minutes each.


### Step 7: Transcribing Chunks with Timestamps

In [13]:
def transcribe_chunks(chunks: list[dict], prompt: str = None) -> list[dict]:
    """Transcribe each audio chunk and adjust timestamps to the full timeline.

    Args:
        chunks: Output of chunk_audio().
        prompt: Optional Whisper context prompt.

    Returns:
        List of segment dicts: {start, end, text}.
    """
    all_segments = []

    for chunk in chunks:
        offset_s = chunk["start_ms"] / 1000
        print(f"🤖 Chunk {chunk['index']+1}: transcribing {chunk['path']} (offset {offset_s:.0f}s)...")

        transcript = transcribe_file(chunk["path"], prompt=prompt)

        if hasattr(transcript, "segments") and transcript.segments:
            for seg in transcript.segments:
                all_segments.append({
                    "start": round(seg.start + offset_s, 2),
                    "end"  : round(seg.end   + offset_s, 2),
                    "text" : seg.text.strip(),
                })
        else:
            # Fallback: treat the whole chunk as one segment
            all_segments.append({
                "start": offset_s,
                "end"  : chunk["end_ms"] / 1000,
                "text" : transcript.text.strip(),
            })

    return all_segments


print("🤖 Transcribing all chunks...\n")
merged_segments = transcribe_chunks(chunks, prompt=MEETING_PROMPT)

print(f"\n✅ Total segments: {len(merged_segments)}")
print("\n📝 Preview (first 3 segments):")
for seg in merged_segments[:3]:
    print(f"  [{seg['start']:.2f}s – {seg['end']:.2f}s] {seg['text']}")

🤖 Transcribing all chunks...

🤖 Chunk 1: transcribing audio/chunks/meeting_sample_chunk01.mp3 (offset 0s)...
🤖 Chunk 2: transcribing audio/chunks/meeting_sample_chunk02.mp3 (offset 600s)...
🤖 Chunk 3: transcribing audio/chunks/meeting_sample_chunk03.mp3 (offset 1200s)...

✅ Total segments: 214

📝 Preview (first 3 segments):
  [0.00s – 6.62s] Ladies and gentlemen, thank you for standing by and welcome to the Etrion 2020 year-end report.
  [7.08s – 12.08s] At this time, all participants are in a listen-only mode. After the speaker's presentation,
  [12.30s – 17.24s] there will be a question and answer session. To ask a question during the session, you will need


### Step 8: Exporting with Timestamps

In [14]:
# ──────────────────────────────────────────────────────────────────
# Helper: seconds → SRT timestamp string  (HH:MM:SS,mmm)
# ──────────────────────────────────────────────────────────────────
def srt_time(seconds: float) -> str:
    h  = int(seconds // 3600)
    m  = int((seconds % 3600) // 60)
    s  = int(seconds % 60)
    ms = int((seconds - int(seconds)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def export_txt(segments: list[dict], path: str) -> None:
    """Human-readable transcript with [MM:SS] timestamps."""
    lines = []
    for seg in segments:
        start_fmt = f"{int(seg['start']//60):02d}:{int(seg['start']%60):02d}"
        end_fmt   = f"{int(seg['end']//60):02d}:{int(seg['end']%60):02d}"
        lines.append(f"[{start_fmt} – {end_fmt}]  {seg['text']}")
    Path(path).write_text("\n".join(lines), encoding="utf-8")
    print(f"  ✅ TXT  → {path}")


def export_srt(segments: list[dict], path: str) -> None:
    """SRT subtitle file format."""
    blocks = []
    for i, seg in enumerate(segments, 1):
        block = f"{i}\n{srt_time(seg['start'])} --> {srt_time(seg['end'])}\n{seg['text']}"
        blocks.append(block)
    Path(path).write_text("\n\n".join(blocks), encoding="utf-8")
    print(f"  ✅ SRT  → {path}")


def export_json(segments: list[dict], path: str, meta: dict = None) -> None:
    """JSON export with optional metadata."""
    payload = {"metadata": meta or {}, "segments": segments}
    Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  ✅ JSON → {path}")


# ──────────────────────────────────────────────────────────────────
# Run exports
# ──────────────────────────────────────────────────────────────────
STEM = Path(AUDIO_FILE).stem

print("💾 Exporting transcriptions...\n")
export_txt(merged_segments,  f"transcripts/{STEM}.txt")
export_srt(merged_segments,  f"transcripts/{STEM}.srt")
export_json(merged_segments, f"transcripts/{STEM}.json",
            meta={"source": AUDIO_FILE, "prompt_used": bool(MEETING_PROMPT)})

print("\n📂 All exports saved to transcripts/")

💾 Exporting transcriptions...

  ✅ TXT  → transcripts/meeting_sample.txt
  ✅ SRT  → transcripts/meeting_sample.srt
  ✅ JSON → transcripts/meeting_sample.json

📂 All exports saved to transcripts/
